# UD4.06 — Del gráfico que engaña al gráfico honesto

**Módulo 5073 · Programación de Inteligencia Artificial · Curso 2026/27**
UD4 — Visualización de datos · 14 horas

Criterios 1.d y 2.e · Ensayo guiado antes de las prácticas P4.1 y P4.2

## De qué va este cuaderno

Los cuatro primeros cuadernos enseñan a dibujar y el quinto a evaluar. Este trata la
cuestión que atraviesa los cinco y que decide si el trabajo sirve de algo:

> **Un gráfico no es una ilustración de los datos: es un argumento sobre los datos.**

Cada decisión que se toma al dibujar —dónde empieza el eje, qué se agrega, qué tramo
se enseña, qué se codifica en el color— es una afirmación. Y como los gráficos
convencen mucho más que las tablas, una afirmación falsa dentro de un gráfico pasa
desapercibida mucho más tiempo.

La forma de aprender esto no es leer una lista de buenas prácticas: es **fabricar los
engaños**. Igual que en la UD3 la fuga de información se aprendía provocándola a
propósito, aquí cada apartado dibuja **dos veces los mismos datos**: una engañando y
otra sin engañar. La comparación es la lección.

Al final hay una lista de comprobación de nueve puntos que se aplica a todos los
gráficos de las prácticas P4.1 y P4.2.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(20262027)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
plt.rcParams["figure.dpi"] = 110

print("numpy", np.__version__, "· pandas", pd.__version__)

In [ ]:
# Solo en Google Colab: descarga el fichero de datos de la unidad.
import os
import urllib.request

BASE = ("https://raw.githubusercontent.com/RafaSalaEsteve/IABD-PIA-notebooks"
        "/main/UD4/datos/")
FICHERO = "ecommerce_ventas_2024.csv"

if not os.path.exists(os.path.join("datos", FICHERO)):
    os.makedirs("datos", exist_ok=True)
    urllib.request.urlretrieve(BASE + FICHERO, os.path.join("datos", FICHERO))
    print("descargado:", FICHERO)
else:
    print("El fichero ya está en datos/, no descargo nada")

# La misma limpieza del cuaderno 03, resumida.
ventas = pd.read_csv(os.path.join("datos", FICHERO))
ventas["Fecha"] = pd.to_datetime(ventas["Fecha"])
ventas["Categoria"] = (ventas["Categoria"].str.strip().str.lower()
                       .str.normalize("NFKD")
                       .str.encode("ascii", "ignore").str.decode("ascii"))
ventas = ventas[(ventas["Precio_Unitario"] > 0)
                & ventas["Cantidad"].between(1, 50)
                & (ventas["Descuento_%"].between(0, 100)
                   | ventas["Descuento_%"].isna())
                & ventas["Region"].notna()].copy()
ventas["Descuento_%"] = ventas["Descuento_%"].fillna(0.0)
ventas["Importe"] = (ventas["Precio_Unitario"] * ventas["Cantidad"]
                     * (1 - ventas["Descuento_%"] / 100) + ventas["Costo_Envio"])
ventas["Mes"] = ventas["Fecha"].dt.to_period("M").dt.to_timestamp()
ventas = ventas.reset_index(drop=True)

print(f"TechStore limpio: {len(ventas)} transacciones")

## 1. El eje que no empieza en cero

Es la manipulación más común del mundo y la más fácil de hacer sin querer, porque
Matplotlib la hace **por defecto**: ajusta los límites a los datos.

La regla no es «el eje siempre empieza en cero». Es más precisa:

> **Si la longitud del elemento gráfico es lo que se lee, el eje tiene que empezar en
> cero.** En una barra se lee la longitud, así que sí. En una línea se lee la
> pendiente, no la altura, así que no hace falta.

Por eso truncar el eje de un gráfico de líneas de temperaturas es normal, y truncar
el de un gráfico de barras es un engaño.

In [ ]:
resumen = (ventas.groupby("Region", as_index=False)["Importe"].mean()
           .sort_values("Importe", ascending=False))
regiones = resumen["Region"].to_numpy()
importes = resumen["Importe"].to_numpy()

fig, (izq, der) = plt.subplots(1, 2, figsize=(15, 5.2))

# El engaño: eje que empieza justo por debajo del mínimo.
izq.bar(regiones, importes, color="#c0392b", edgecolor="black", linewidth=0.6)
izq.set_ylim(importes.min() - 2, importes.max() + 2)
izq.set_title("«Madrid factura MUCHO más que Canarias»", fontweight="bold",
              fontsize=12, color="#922b21")
izq.set_ylabel("Importe medio (€)")
izq.tick_params(axis="x", rotation=55, labelsize=8)
izq.grid(True, alpha=0.3, axis="y")

# El honesto: el mismo dato desde cero.
der.bar(regiones, importes, color="#1d6b3f", edgecolor="black", linewidth=0.6)
der.set_ylim(0, importes.max() * 1.12)
der.set_title(f"«Las siete regiones facturan casi lo mismo: entre "
              f"{importes.min():.0f} y {importes.max():.0f} €»",
              fontweight="bold", fontsize=12, color="#145a32")
der.set_ylabel("Importe medio (€)")
der.tick_params(axis="x", rotation=55, labelsize=8)
der.grid(True, alpha=0.3, axis="y")
der.bar_label(der.containers[0], fmt="%.0f", fontsize=7, padding=2)

fig.suptitle("Los mismos siete números, dos titulares incompatibles",
             fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

razon_real = importes.max() / importes.min()
altura_izq = ((importes.max() - (importes.min() - 2))
              / (importes[-1] - (importes.min() - 2)))
print(f"La diferencia REAL entre la primera región y la última: "
      f"{razon_real:.2f} veces ({(razon_real - 1) * 100:.0f} %).")
print(f"La razón entre las ALTURAS DIBUJADAS en el gráfico de la izquierda: "
      f"{altura_izq:.0f} veces.")
print()
print(f"O sea que el gráfico de la izquierda exagera la diferencia unas "
      f"{altura_izq / razon_real:.0f} veces.")
print()
print("Y no ha hecho falta manipular ni un dato: los siete números son los mismos.")
print("Solo se ha cambiado el límite inferior del eje Y, que es UNA línea de código,")
print("y encima es la que Matplotlib pone por defecto si no se dice nada.")
print()
print("Por eso `ax.set_ylim(0, ...)` en un gráfico de barras no es una precaución:")
print("es parte de dibujarlo bien.")

### 1.1 Cuándo truncar sí es correcto

La regla tiene su otra mitad, y es igual de importante: **empezar en cero cuando no
corresponde también engaña**, porque aplasta la variación que se quería enseñar.

In [ ]:
temperatura = 21.5 + 1.4 * np.sin(np.arange(24) / 3.5) + rng.normal(0, 0.25, 24)
horas = np.arange(24)

fig, (izq, der) = plt.subplots(1, 2, figsize=(15, 4.6))

izq.plot(horas, temperatura, "o-", color="#1d6b3f", linewidth=2)
izq.set_ylim(0, 30)
izq.set_title("Con el eje desde cero: parece que no pasa nada",
              fontweight="bold", fontsize=11, color="#922b21")
izq.set_xlabel("Hora del día")
izq.set_ylabel("Temperatura del servidor (°C)")
izq.grid(True, alpha=0.3)

der.plot(horas, temperatura, "o-", color="#1d6b3f", linewidth=2)
der.axhspan(20, 23, alpha=0.12, color="#1d6b3f")
der.text(0.4, 22.7, "rango operativo recomendado", fontsize=8, color="#145a32")
der.set_title("Con el eje ajustado: se ve el ciclo de 7 horas",
              fontweight="bold", fontsize=11, color="#145a32")
der.set_xlabel("Hora del día")
der.set_ylabel("Temperatura del servidor (°C)")
der.grid(True, alpha=0.3)

fig.suptitle("Aquí truncar es lo correcto: en una línea se lee la PENDIENTE",
             fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

print(f"La temperatura oscila entre {temperatura.min():.1f} y "
      f"{temperatura.max():.1f} °C, {temperatura.max() - temperatura.min():.1f} grados.")
print()
print("En el gráfico de la izquierda esa oscilación es indistinguible de una línea")
print("recta, y es exactamente la información por la que se hizo el gráfico.")
print()
print("La regla, otra vez: la longitud pide cero, la pendiente no. Y en los dos casos")
print("hay que ser capaz de decir POR QUÉ se ha elegido el límite del eje.")

## 2. Los sectores

El diagrama de sectores codifica la magnitud en el **ángulo**, y el ojo humano compara
ángulos mucho peor que longitudes. Está medido desde los años ochenta: en los
experimentos de Cleveland y McGill, la posición sobre una escala común es la
codificación más precisa y el ángulo está muy por detrás.

La consecuencia práctica:

| Situación | Qué usar |
|---|---|
| Dos o tres partes de un total, con proporciones muy distintas | Sectores, si se quiere |
| Cualquier otra cosa | **Barras**, y ordenadas de mayor a menor |

Y hay una condición que se olvida: los sectores solo tienen sentido si las partes
**suman un total con significado**. Un diagrama de sectores de las notas de seis
alumnos no significa nada, porque las notas no son partes de nada.

In [ ]:
por_categoria = (ventas.groupby("Categoria")["Importe"].sum()
                 .sort_values(ascending=False))
etiquetas = por_categoria.index.to_numpy()
valores = por_categoria.to_numpy()

fig = plt.figure(figsize=(16, 5))
gs = fig.add_gridspec(1, 3, wspace=0.35)

# El engaño 1: sectores con seis porciones y sin orden.
ax = fig.add_subplot(gs[0, 0])
desorden = rng.permutation(len(valores))
ax.pie(valores[desorden], labels=etiquetas[desorden], autopct="%1.1f%%",
       startangle=17, textprops={"fontsize": 7})
ax.set_title("Sectores, sin ordenar\n¿Cuál es la tercera categoría?",
             fontweight="bold", fontsize=11, color="#922b21")

# El engaño 2: el mismo, pero con las porciones separadas y en otro orden. Parece
# otro reparto, y son exactamente los mismos números.
ax = fig.add_subplot(gs[0, 1])
otro_desorden = rng.permutation(len(valores))
ax.pie(valores[otro_desorden], labels=etiquetas[otro_desorden],
       autopct="%1.1f%%", startangle=200,
       explode=[0.06] * len(valores), textprops={"fontsize": 7})
ax.set_title("Los MISMOS números, otro ángulo de partida\n"
             "Se lee como si fuera otro reparto",
             fontweight="bold", fontsize=11, color="#922b21")

# El honesto: barras horizontales, ordenadas, con el porcentaje escrito.
ax = fig.add_subplot(gs[0, 2])
total = valores.sum()
posicion = np.arange(len(valores))[::-1]
barras = ax.barh(posicion, valores, color="#1d6b3f", edgecolor="black",
                 linewidth=0.5)
ax.set_yticks(posicion, etiquetas, fontsize=9)
ax.set_xlabel("Importe total (€)")
ax.set_xlim(0, valores.max() * 1.22)
for y, valor in zip(posicion, valores):
    ax.text(valor * 1.02, y, f"{valor / total:.1%}", va="center", fontsize=8,
            fontweight="bold")
ax.set_title("Barras ordenadas\nSe lee el orden, la magnitud y la proporción",
             fontweight="bold", fontsize=11, color="#145a32")
ax.grid(True, alpha=0.3, axis="x")

fig.suptitle("Seis categorías: el mismo reparto de tres formas", fontsize=14,
             fontweight="bold")
plt.show()

print("Prueba a contestar estas tres preguntas mirando SOLO el primer gráfico:")
print()
print("  1. ¿Cuál es la tercera categoría por importe?")
print("  2. ¿Cuánto más grande es la primera que la segunda?")
print("  3. ¿Suman las tres últimas más o menos que la primera?")
print()
print("Ahora contéstalas mirando el tercero. Las tres son inmediatas.")
print()
print("Las respuestas, para comprobar:")
print(f"  1. {etiquetas[2]}")
print(f"  2. {valores[0] / valores[1]:.2f} veces")
tres_ultimas = valores[-3:].sum()
print(f"  3. Las tres últimas suman {tres_ultimas:,.0f} € y la primera "
      f"{valores[0]:,.0f} €: "
      f"{'MENOS' if tres_ultimas < valores[0] else 'MÁS'}")
print()
print("Y el porcentaje escrito en los sectores es la confesión del formato: si hay")
print("que escribir el número porque el ángulo no se lee, el ángulo no estaba")
print("aportando nada.")

## 3. El área que no es proporcional

Cuando la magnitud se codifica en el tamaño de una forma, hay que decidir si el valor
es proporcional al **radio** o al **área**. El ojo percibe el área, así que la
codificación correcta es el área. Poner el valor en el radio multiplica la
exageración por el cuadrado.

Es el error del gráfico de burbujas, y también el del icono que se «escala» en una
infografía.

In [ ]:
valores_muestra = np.array([10, 20, 40, 80])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.6))

# El engaño: el radio proporcional al valor.
ax = axes[0]
for i, v in enumerate(valores_muestra):
    ax.scatter(i, 0, s=(v * 5.5) ** 2 / 60, color="#c0392b", alpha=0.75,
               edgecolors="black", linewidth=0.8)
    ax.text(i, -0.55, f"{v}", ha="center", fontsize=11, fontweight="bold")
ax.set_xlim(-0.7, 3.7)
ax.set_ylim(-0.8, 0.8)
ax.axis("off")
ax.set_title("RADIO proporcional al valor\nEl último parece 64 veces el primero",
             fontweight="bold", fontsize=11, color="#922b21")

# El correcto: el área proporcional al valor.
ax = axes[1]
for i, v in enumerate(valores_muestra):
    ax.scatter(i, 0, s=v * 42, color="#1d6b3f", alpha=0.75,
               edgecolors="black", linewidth=0.8)
    ax.text(i, -0.55, f"{v}", ha="center", fontsize=11, fontweight="bold")
ax.set_xlim(-0.7, 3.7)
ax.set_ylim(-0.8, 0.8)
ax.axis("off")
ax.set_title("ÁREA proporcional al valor\nEl último es 8 veces el primero, y lo parece",
             fontweight="bold", fontsize=11, color="#145a32")

# Lo que se debería haber hecho desde el principio.
ax = axes[2]
ax.bar(range(len(valores_muestra)), valores_muestra, color="#1a5276",
       edgecolor="black", linewidth=0.6)
ax.set_xticks(range(len(valores_muestra)),
              [str(v) for v in valores_muestra])
ax.set_ylim(0, valores_muestra.max() * 1.15)
ax.set_ylabel("Valor")
ax.set_xlabel("Valor")
ax.set_title("O directamente barras\nLa longitud se lee sin ambigüedad",
             fontweight="bold", fontsize=11, color="#145a32")
ax.grid(True, alpha=0.3, axis="y")

fig.suptitle("Cuatro números —10, 20, 40, 80— con tres codificaciones",
             fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

print("La aritmética del engaño:")
print()
print(f"  El valor pasa de {valores_muestra[0]} a {valores_muestra[-1]}: "
      f"{valores_muestra[-1] / valores_muestra[0]:.0f} veces.")
print(f"  Si el RADIO es proporcional al valor, el área crece "
      f"{(valores_muestra[-1] / valores_muestra[0]) ** 2:.0f} veces,")
print("  y el área es lo que el ojo percibe.")
print()
print("En Matplotlib el parámetro `s` de `scatter` es el ÁREA del marcador en puntos")
print("cuadrados, así que `s=valor*k` está bien y `s=(valor*k)**2` está mal. Es una")
print("de esas cosas que hay que saber una vez y no olvidar.")

## 4. La ventana escogida

Enseñar el tramo de datos que respalda la conclusión que ya se tenía. No hay ningún
dato falso: hay un `[inicio:fin]` bien elegido.

Es el más difícil de detectar de los siete, porque desde fuera **no se ve que falte
nada**. La única defensa es una norma de proceso: enseñar siempre la serie completa,
y si se destaca un tramo, destacarlo **dentro** de la serie completa.

In [ ]:
por_mes = ventas.groupby("Mes", as_index=False)["Importe"].sum()
meses = por_mes["Mes"].to_numpy()
totales = por_mes["Importe"].to_numpy()

# Buscamos el tramo de cuatro meses de mayor crecimiento y el de mayor caída, que es
# lo que haría quien quiere contar una historia concreta.
ventana = 4
crecimientos = [(totales[i + ventana - 1] / totales[i] - 1, i)
                for i in range(len(totales) - ventana + 1)]
mejor_subida, i_subida = max(crecimientos)
peor_bajada, i_bajada = min(crecimientos)

fig = plt.figure(figsize=(16, 8))
gs = fig.add_gridspec(2, 2, hspace=0.42, wspace=0.25)

ax = fig.add_subplot(gs[0, 0])
tramo = slice(i_subida, i_subida + ventana)
ax.plot(meses[tramo], totales[tramo], "o-", color="#1d6b3f", linewidth=3,
        markersize=9)
ax.set_title(f"«Las ventas crecen un {mejor_subida:.0%} en cuatro meses»",
             fontweight="bold", fontsize=12, color="#145a32")
ax.set_ylabel("Importe total (€)")
ax.tick_params(axis="x", rotation=25, labelsize=8)
ax.grid(True, alpha=0.3)

ax = fig.add_subplot(gs[0, 1])
tramo = slice(i_bajada, i_bajada + ventana)
ax.plot(meses[tramo], totales[tramo], "o-", color="#c0392b", linewidth=3,
        markersize=9)
ax.set_title(f"«Las ventas se hunden un {abs(peor_bajada):.0%} en cuatro meses»",
             fontweight="bold", fontsize=12, color="#922b21")
ax.set_ylabel("Importe total (€)")
ax.tick_params(axis="x", rotation=25, labelsize=8)
ax.grid(True, alpha=0.3)

ax = fig.add_subplot(gs[1, :])
ax.plot(meses, totales, "o-", color="#1a5276", linewidth=2.4, markersize=7,
        label="Importe mensual")
ax.axhline(totales.mean(), color="0.45", linestyle="--", linewidth=1.6,
           label=f"media = {totales.mean():,.0f} €")
ax.axvspan(meses[i_subida], meses[i_subida + ventana - 1], alpha=0.14,
           color="#1d6b3f")
ax.axvspan(meses[i_bajada], meses[i_bajada + ventana - 1], alpha=0.14,
           color="#c0392b")
ax.text(meses[i_subida + 1], totales.min() * 0.97, "el tramo «crecemos»",
        fontsize=9, color="#145a32", fontweight="bold")
ax.text(meses[i_bajada + 1], totales.min() * 0.97, "el tramo «nos hundimos»",
        fontsize=9, color="#922b21", fontweight="bold")
ax.set_title("La serie completa: ni crece ni se hunde. Oscila alrededor de la media",
             fontweight="bold", fontsize=12)
ax.set_xlabel("Mes")
ax.set_ylabel("Importe total (€)")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

fig.suptitle("Dos titulares opuestos, sacados de la misma serie de doce meses",
             fontsize=14, fontweight="bold")
plt.show()

variacion_total = totales[-1] / totales[0] - 1
print(f"Los doce meses, de enero a diciembre: {variacion_total:+.1%}")
print(f"Oscilación alrededor de la media: ±"
      f"{np.abs(totales - totales.mean()).max() / totales.mean():.1%}")
print()
print(f"Y con la misma serie se pueden titular un {mejor_subida:+.0%} y un "
      f"{peor_bajada:+.0%}, sin tocar")
print("ni un dato: solo eligiendo los cuatro meses que se enseñan.")
print()
print("La defensa, que es de proceso y no de técnica:")
print()
print("  1. Enseñar la serie COMPLETA, siempre, aunque el mensaje sea de un tramo.")
print("  2. Si hay que destacar un tramo, destacarlo DENTRO de la serie completa,")
print("     con `axvspan`, como en el panel de abajo.")
print("  3. Poner una referencia: la media, el año anterior, el objetivo. Sin una")
print("     referencia, cualquier variación puede parecer grande o pequeña.")
print("  4. Con doce puntos y esta oscilación, la conclusión honesta es que NO hay")
print("     tendencia detectable. Y decir «no hay señal» es un resultado, no un")
print("     fracaso.")

## 5. La media que esconde la dispersión

Agregar es imprescindible: sin agregar no se puede resumir nada. Pero **cada
agregación tira información, y hay que saber cuál**.

Una barra con la media dice «este grupo vale esto». Si dentro del grupo hay dos
poblaciones distintas, o una cola larga, la barra afirma algo que no es verdad, y no
hay forma de saberlo mirándola.

In [ ]:
# Tres grupos con la MISMA media y repartos que no se parecen en nada.
n = 400
grupos = {
    "Homogéneo": rng.normal(100, 6, n),
    "Dos poblaciones": np.concatenate([rng.normal(70, 7, n // 2),
                                       rng.normal(130, 7, n // 2)]),
    "Con cola larga": np.concatenate([rng.normal(84, 6, int(n * 0.9)),
                                      rng.normal(244, 30, n - int(n * 0.9))]),
}
# Ajustamos para que las tres medias coincidan exactamente.
for nombre in grupos:
    grupos[nombre] = grupos[nombre] - grupos[nombre].mean() + 100.0

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

# El engaño: tres barras iguales.
ax = axes[0]
medias = [g.mean() for g in grupos.values()]
ax.bar(range(3), medias, color="#c0392b", edgecolor="black", linewidth=0.6)
ax.set_xticks(range(3), list(grupos), fontsize=9)
ax.set_ylim(0, 130)
ax.set_ylabel("Valor medio")
ax.set_title("Tres barras idénticas\n«Los tres grupos son iguales»",
             fontweight="bold", fontsize=11, color="#922b21")
ax.grid(True, alpha=0.3, axis="y")
ax.bar_label(ax.containers[0], fmt="%.1f", fontsize=9, padding=3)

# Medio arreglo: barras con la desviación. Mejor, y todavía insuficiente.
ax = axes[1]
desviaciones = [g.std() for g in grupos.values()]
ax.bar(range(3), medias, yerr=desviaciones, capsize=7, color="#e67e22",
       edgecolor="black", linewidth=0.6)
ax.set_xticks(range(3), list(grupos), fontsize=9)
ax.set_ylim(0, 180)
ax.set_ylabel("Valor medio ± desviación")
ax.set_title("Con la desviación: se ve que NO son iguales\n"
             "pero no se ve en qué se diferencian",
             fontweight="bold", fontsize=11, color="#7e5109")
ax.grid(True, alpha=0.3, axis="y")

# El honesto: la forma completa.
ax = axes[2]
partes = ax.violinplot(list(grupos.values()), positions=range(3),
                       showmedians=True, widths=0.8)
for cuerpo in partes["bodies"]:
    cuerpo.set_facecolor("#1d6b3f")
    cuerpo.set_alpha(0.55)
for i, valores in enumerate(grupos.values()):
    muestra = rng.choice(valores, 120, replace=False)
    ax.scatter(np.full(len(muestra), i) + rng.normal(0, 0.045, len(muestra)),
               muestra, s=5, color="black", alpha=0.35)
ax.set_xticks(range(3), list(grupos), fontsize=9)
ax.set_ylabel("Valor")
ax.set_title("La forma completa\nAhora sí se ve qué hay dentro de cada grupo",
             fontweight="bold", fontsize=11, color="#145a32")
ax.grid(True, alpha=0.3, axis="y")

fig.suptitle("Tres grupos con la MISMA media (100,0)", fontsize=14,
             fontweight="bold")
fig.tight_layout()
plt.show()

print(f"{'grupo':>18} {'media':>8} {'mediana':>9} {'desv.':>8} "
      f"{'P5':>8} {'P95':>8}")
print("-" * 64)
for nombre, valores in grupos.items():
    print(f"{nombre:>18} {valores.mean():>8.1f} {np.median(valores):>9.1f} "
          f"{valores.std():>8.1f} {np.percentile(valores, 5):>8.1f} "
          f"{np.percentile(valores, 95):>8.1f}")

print()
print("Fíjate en «Dos poblaciones»: la media es 100 y la mediana también, y NO HAY")
print("PRÁCTICAMENTE NADIE cerca de 100. Es el valor típico de un grupo en el que ese")
print("valor no le pasa a casi nadie.")
print()
print("Y en «Con cola larga»: la mediana es 16 puntos más baja que la media, porque el")
print("10 % de la cola arrastra el promedio. Informar de la media aquí describe un")
print("caso que no es el habitual.")
print()
print("La regla práctica: **una barra con una media necesita justificación**. Si el")
print("reparto no es unimodal y aproximadamente simétrico, la media no resume, y hay")
print("que enseñar la forma o al menos la mediana y los percentiles.")

## 6. El agregado que dice lo contrario que los grupos

Este es el más serio de los siete, porque **no es un problema de dibujo sino de
razonamiento**, y afecta también a las tablas. Tiene nombre propio: **la paradoja de
Simpson**.

Ocurre así: se comparan dos grupos en el total y gana A. Se desglosa por una tercera
variable y **gana B en todos y cada uno de los subgrupos**. Las dos afirmaciones son
aritméticamente correctas.

El caso que sigue está fabricado a propósito y con números redondos, para que se vea
la mecánica. En los datos reales aparece continuamente y casi nunca se busca.

In [ ]:
# Dos canales de venta, dos categorías de producto.
#
# El canal WEB vende sobre todo informática, que es cara. El canal MÓVIL vende sobre
# todo accesorios, que son baratos. Dentro de CADA categoría, el móvil consigue un
# ticket medio mayor.
casos = []
configuracion = [
    ("web",   "informatica", 900,  480.0),
    ("web",   "accesorios",  100,   62.0),
    ("movil", "informatica", 120,  520.0),
    ("movil", "accesorios",  880,   68.0),
]
for canal, categoria, cuantos, ticket in configuracion:
    importes = rng.normal(ticket, ticket * 0.12, cuantos)
    casos.append(pd.DataFrame({"canal": canal, "categoria": categoria,
                               "importe": importes}))
pedidos = pd.concat(casos, ignore_index=True)

global_por_canal = pedidos.groupby("canal")["importe"].mean()
por_canal_y_categoria = (pedidos.groupby(["categoria", "canal"])["importe"]
                         .mean().unstack())
reparto = (pd.crosstab(pedidos["canal"], pedidos["categoria"], normalize="index"))

print("EN EL TOTAL:")
for canal, valor in global_por_canal.items():
    print(f"  {canal:>6}: ticket medio {valor:>7.2f} €")
ganador_global = global_por_canal.idxmax()
print(f"  → gana {ganador_global.upper()}")
print()
print("DESGLOSADO POR CATEGORÍA:")
print(por_canal_y_categoria.round(2).to_string())
print()
for categoria, fila in por_canal_y_categoria.iterrows():
    print(f"  en {categoria:>12}: gana {fila.idxmax().upper()} "
          f"({fila.max():.2f} € frente a {fila.min():.2f} €)")
print()
print("Las dos cosas son verdad a la vez, y no hay ningún error de cálculo.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16.5, 5))

colores_canal = {"web": "#1a5276", "movil": "#c0392b"}

# El engaño: solo el total.
ax = axes[0]
ax.bar(global_por_canal.index, global_por_canal.to_numpy(),
       color=[colores_canal[c] for c in global_por_canal.index],
       edgecolor="black", linewidth=0.6)
ax.set_ylabel("Ticket medio (€)")
ax.set_ylim(0, global_por_canal.max() * 1.2)
ax.bar_label(ax.containers[0], fmt="%.0f €", fontsize=10, padding=3)
ax.set_title(f"Solo el total\n«{ganador_global.upper()} vende mejor»",
             fontweight="bold", fontsize=11, color="#922b21")
ax.grid(True, alpha=0.3, axis="y")

# El honesto: desglosado.
ax = axes[1]
posicion = np.arange(len(por_canal_y_categoria))
ancho = 0.36
for i, canal in enumerate(por_canal_y_categoria.columns):
    ax.bar(posicion + (i - 0.5) * ancho, por_canal_y_categoria[canal],
           ancho, label=canal, color=colores_canal[canal],
           edgecolor="black", linewidth=0.6)
ax.set_xticks(posicion, por_canal_y_categoria.index)
ax.set_ylabel("Ticket medio (€)")
ax.legend(title="canal", fontsize=9)
ax.set_title("Desglosado por categoría\n«MÓVIL vende mejor en las DOS»",
             fontweight="bold", fontsize=11, color="#145a32")
ax.grid(True, alpha=0.3, axis="y")

# La explicación: el reparto de categorías dentro de cada canal.
ax = axes[2]
abajo = np.zeros(len(reparto))
for categoria in reparto.columns:
    ax.bar(reparto.index, reparto[categoria], bottom=abajo, label=categoria,
           edgecolor="black", linewidth=0.6)
    for i, (canal, valor) in enumerate(reparto[categoria].items()):
        if valor > 0.06:
            ax.text(i, abajo[i] + valor / 2, f"{valor:.0%}", ha="center",
                    va="center", fontsize=10, fontweight="bold", color="white")
    abajo = abajo + reparto[categoria].to_numpy()
ax.set_ylabel("Proporción de pedidos")
ax.set_ylim(0, 1)
ax.legend(title="categoría", fontsize=9, loc="lower right")
ax.set_title("LA CAUSA: qué vende cada canal\nEl 90 % de la web es informática",
             fontweight="bold", fontsize=11)

fig.suptitle("La paradoja de Simpson: el total dice lo contrario que los grupos",
             fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

print("La mecánica, en dos líneas:")
print()
print("El canal web vende sobre todo informática, que tiene un ticket alto. El móvil")
print("vende sobre todo accesorios, que lo tienen bajo. Al promediar sin desglosar, lo")
print("que se compara NO es la calidad de cada canal: es la mezcla de productos que")
print("vende cada uno.")
print()
print("La variable «categoría» está detrás de las dos que se comparan, y por eso se")
print("llama VARIABLE DE CONFUSIÓN. Mientras no se controle, el gráfico del total no")
print("contesta a la pregunta «¿qué canal vende mejor?».")
print()
print("Y la parte incómoda: no hay ninguna técnica de visualización que lo detecte")
print("solo. Hay que SOSPECHARLO y desglosar. La norma de trabajo:")
print()
print("  Antes de publicar una comparación entre dos grupos, desglósala por las dos o")
print("  tres variables que puedan estar detrás. Si el orden cambia en alguna, el")
print("  gráfico del total NO se publica solo.")
print()
print("En TechStore las variables candidatas son evidentes: categoría, región y mes.")
print("Es el ejercicio 4 de este cuaderno.")

## 7. El acumulado

Una serie acumulada **siempre sube**, porque cada punto es la suma de todo lo
anterior. Eso hace que un gráfico de acumulados nunca pueda mostrar una caída, y por
eso es la forma preferida de enseñar un negocio que se está frenando.

La pregunta que hay que hacerse: ¿se está informando del **nivel** o de la
**variación**? El acumulado responde al nivel; la variación necesita la serie sin
acumular, o la derivada.

In [ ]:
# Un producto cuyas ventas mensuales CAEN sistemáticamente desde el mes 5.
mensual = np.array([100, 122, 141, 158, 168, 160, 145, 128, 110, 94, 78, 65],
                   dtype=float)
acumulado = np.cumsum(mensual)
etiquetas_mes = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
                 "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.8))

ax = axes[0]
ax.plot(etiquetas_mes, acumulado, "o-", color="#c0392b", linewidth=3, markersize=7)
ax.fill_between(range(12), 0, acumulado, alpha=0.15, color="#c0392b")
ax.set_ylabel("Usuarios acumulados")
ax.set_ylim(0, acumulado.max() * 1.1)
ax.set_title("«Crecimiento imparable: 1.369 usuarios»\nSube todos los meses",
             fontweight="bold", fontsize=11, color="#922b21")
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.bar(etiquetas_mes, mensual, color="#1d6b3f", edgecolor="black", linewidth=0.5)
ax.axvline(4.5, color="black", linestyle="--", linewidth=1.6)
ax.text(5.0, mensual.max() * 0.95, "desde junio, cae\ntodos los meses",
        fontsize=9, color="#922b21", fontweight="bold")
ax.set_ylabel("Usuarios nuevos en el mes")
ax.set_title("Los MISMOS datos, sin acumular\nEl máximo fue en mayo",
             fontweight="bold", fontsize=11, color="#145a32")
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.grid(True, alpha=0.3, axis="y")

ax = axes[2]
variacion = np.diff(mensual, prepend=mensual[0]) / mensual[0] * 100
ax.bar(etiquetas_mes, variacion,
       color=np.where(variacion >= 0, "#1d6b3f", "#c0392b"),
       edgecolor="black", linewidth=0.5)
ax.axhline(0, color="black", linewidth=1.2)
ax.set_ylabel("Variación respecto al mes anterior (%)")
ax.set_title("Y la variación mes a mes\nSiete meses seguidos en negativo",
             fontweight="bold", fontsize=11, color="#145a32")
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.grid(True, alpha=0.3, axis="y")

fig.suptitle("Acumulado, mensual y variación: los mismos doce números",
             fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

print(f"Total acumulado en el año: {acumulado[-1]:,.0f} usuarios.")
print(f"Usuarios nuevos en mayo:  {mensual[4]:,.0f}")
print(f"Usuarios nuevos en diciembre: {mensual[-1]:,.0f} "
      f"({mensual[-1] / mensual[4] - 1:+.0%} respecto al máximo)")
print()
print("El primer gráfico no tiene ni un dato falso, y es imposible que muestre la")
print("caída: una serie acumulada de cantidades positivas no puede bajar nunca.")
print()
print("Cuándo el acumulado SÍ es lo correcto: cuando la pregunta es de nivel. «Cuántos")
print("usuarios tenemos en total», «cuánto llevamos facturado en el año», «cuántas")
print("vacunas se han puesto». Ahí el acumulado ES la respuesta.")
print()
print("La comprobación rápida: si el gráfico está para responder a «¿vamos bien o")
print("mal?», el acumulado no sirve, porque siempre contesta «bien».")

## 8. La disciplina del rediseño

Los siete apartados anteriores enseñan qué no hacer. Esta sección es el procedimiento
para hacerlo bien, y son tres preguntas en orden. **El orden importa**: la tercera no
se puede contestar antes que la primera.

### Pregunta 1. ¿A qué pregunta responde este gráfico?

Una sola pregunta, escrita en una frase y con un signo de interrogación. Si no se
puede escribir, el gráfico todavía no se puede dibujar, porque no se sabe qué tiene
que enseñar.

La prueba de fuego: **el título del gráfico acabado tiene que ser la respuesta a esa
pregunta**, no el nombre de las variables. «Importe por región» no es un título:
es una etiqueta. «Las siete regiones facturan lo mismo» sí lo es.

### Pregunta 2. ¿Qué codificación responde mejor a esa pregunta?

Del trabajo de Cleveland y McGill, ordenado de más preciso a menos:

| | Codificación | Se lee |
|---|---|---|
| 1 | Posición en una escala común | Muy bien |
| 2 | Posición en escalas alineadas | Bien |
| 3 | Longitud | Bien |
| 4 | Ángulo, pendiente | Regular |
| 5 | Área | Mal |
| 6 | Volumen, curvatura | Muy mal |
| 7 | Saturación de color, tono | Solo para categorías o para el fondo |

La regla que se deduce: **la variable más importante va en la codificación más
precisa que quede libre.** Si lo que importa es comparar magnitudes, van en el eje;
lo secundario va al color.

### Pregunta 3. ¿Qué se puede quitar sin perder la respuesta?

Y aquí se quita: rejillas gruesas, bordes que no informan, decimales que nadie
necesita, leyendas que se pueden sustituir por una etiqueta junto a la línea,
tres dimensiones en algo que es plano, degradados.

El criterio no es el minimalismo por gusto: **cada elemento que no ayuda a responder
la pregunta compite por la atención con los que sí ayudan**.

In [ ]:
# El rediseño completo, paso a paso, sobre una pregunta de TechStore.
#
# PREGUNTA 1: ¿qué categorías concentran el gasto, y cuánto se lleva la primera?

por_categoria = (ventas.groupby("Categoria")["Importe"]
                 .agg(total="sum", transacciones="size", mediana="median")
                 .sort_values("total", ascending=False))

fig = plt.figure(figsize=(16, 9.5))
gs = fig.add_gridspec(2, 2, hspace=0.42, wspace=0.28)

# PASO 0: lo que sale por defecto y sin pensar.
ax = fig.add_subplot(gs[0, 0])
ax.bar(por_categoria.index, por_categoria["total"],
       color=["red", "lime", "blue", "yellow", "magenta", "cyan"])
ax.set_title("Paso 0: lo que sale sin pensar", fontweight="bold", fontsize=11,
             color="#922b21")
ax.tick_params(axis="x", rotation=45, labelsize=8)
ax.grid(True)

# PASO 1: la codificación correcta. Barras horizontales ordenadas, porque las
# etiquetas son texto largo y la comparación es de magnitudes.
ax = fig.add_subplot(gs[0, 1])
posicion = np.arange(len(por_categoria))[::-1]
ax.barh(posicion, por_categoria["total"], color="#1a5276")
ax.set_yticks(posicion, por_categoria.index, fontsize=9)
ax.set_xlabel("Importe total (€)")
ax.set_title("Paso 1: codificación correcta\nlongitud, ordenada, horizontal",
             fontweight="bold", fontsize=11)
ax.grid(True, alpha=0.3, axis="x")

# PASO 2: quitar lo que no responde y añadir lo que falta.
ax = fig.add_subplot(gs[1, 0])
ax.barh(posicion, por_categoria["total"], color="#bdc3c7", height=0.65)
ax.barh(posicion[0], por_categoria["total"].iloc[0], color="#1a5276", height=0.65)
ax.set_yticks(posicion, por_categoria.index, fontsize=9)
ax.set_xlabel("Importe total (€)")
ax.set_xlim(0, por_categoria["total"].max() * 1.2)
for y, valor in zip(posicion, por_categoria["total"]):
    ax.text(valor * 1.02, y, f"{valor / 1000:,.0f} k€", va="center", fontsize=8.5)
ax.set_title("Paso 2: destacar la respuesta y quitar el resto\n"
             "gris para el contexto, color para el mensaje",
             fontweight="bold", fontsize=11)
for lado in ("top", "right", "bottom"):
    ax.spines[lado].set_visible(False)
ax.set_xticks([])
ax.grid(False)

# PASO 3: el título es la respuesta, y hay una referencia con la que comparar.
ax = fig.add_subplot(gs[1, 1])
total_general = por_categoria["total"].sum()
proporcion_primera = por_categoria["total"].iloc[0] / total_general
ax.barh(posicion, por_categoria["total"], color="#bdc3c7", height=0.65)
ax.barh(posicion[0], por_categoria["total"].iloc[0], color="#1a5276", height=0.65)
ax.set_yticks(posicion, por_categoria.index, fontsize=9)
ax.set_xlim(0, por_categoria["total"].max() * 1.28)
for y, (categoria, fila) in zip(posicion, por_categoria.iterrows()):
    ax.text(fila["total"] * 1.02, y,
            f"{fila['total'] / 1000:,.0f} k€   ({fila['total'] / total_general:.0%})",
            va="center", fontsize=8.5,
            fontweight="bold" if y == posicion[0] else "normal")
ax.set_title(f"Informática se lleva el {proporcion_primera:.0%} de la facturación,\n"
             f"más que las cuatro últimas categorías juntas",
             fontweight="bold", fontsize=12, loc="left", color="#1a5276")
for lado in ("top", "right", "bottom"):
    ax.spines[lado].set_visible(False)
ax.set_xticks([])
ax.grid(False)
ax.text(0, -0.9, f"TechStore · {len(ventas):,} transacciones de 2024 · "
                 f"importe con descuento y envío incluidos",
        fontsize=7.5, color="0.45", transform=ax.get_yaxis_transform())

fig.suptitle("Los cuatro pasos del rediseño, sobre los mismos seis números",
             fontsize=15, fontweight="bold")
plt.show()

cuatro_ultimas = por_categoria["total"].iloc[-4:].sum()
print(f"La afirmación del título, comprobada:")
print(f"  informática:            {por_categoria['total'].iloc[0]:>12,.2f} € "
      f"({proporcion_primera:.1%})")
print(f"  las cuatro últimas:     {cuatro_ultimas:>12,.2f} €")
print(f"  ¿es informática mayor?  "
      f"{por_categoria['total'].iloc[0] > cuatro_ultimas}")
print()
print("Qué ha cambiado del paso 0 al paso 3, elemento a elemento:")
print()
print("  QUITADO:  seis colores sin significado, la rejilla completa, las etiquetas")
print("            del eje X rotadas, tres de los cuatro bordes, el eje de valores")
print("            (los números están junto a cada barra, que se lee mejor)")
print("  AÑADIDO:  el orden por magnitud, el porcentaje sobre el total, la nota de")
print("            procedencia al pie, y el título con la conclusión")
print("  CAMBIADO: barras verticales por horizontales (las etiquetas son texto")
print("            largo), y el color pasa de decorar a señalar UNA cosa")
print()
print("Y la comprobación final: el título dice una afirmación, y la afirmación se")
print("puede verificar con los datos. Es lo que separa un gráfico de un adorno.")

## 9. La lista de comprobación

Nueve preguntas. Se aplican a **todos** los gráficos de las prácticas P4.1 y P4.2, y
la rúbrica las da por supuestas.

| | Pregunta | Si la respuesta es «no» |
|---|---|---|
| 1 | ¿Puedo escribir en una frase la pregunta a la que responde? | No dibujes todavía |
| 2 | ¿El título es la respuesta, y no el nombre de las variables? | Reescríbelo |
| 3 | ¿Están los dos ejes etiquetados **con su unidad**? | Ponlas |
| 4 | Si hay barras, ¿empieza el eje en cero? Si no hay, ¿puedo justificar el límite? | Arréglalo |
| 5 | ¿La variable importante va en la codificación más precisa disponible? | Cámbialas |
| 6 | ¿Codifica el color una variable, o solo decora? | Quítalo |
| 7 | ¿Estoy enseñando **toda** la serie, o un tramo elegido? | Enseña el resto |
| 8 | Si agrego, ¿he mirado la forma de lo que estoy agregando? | Mírala |
| 9 | Si comparo dos grupos, ¿he desglosado por lo que pueda estar detrás? | Desglósalo |

Y dos más que no son del gráfico sino de la entrega:

10. ¿Consta **de dónde salen los datos** y de cuándo son?
11. ¿Se puede leer impreso en blanco y negro y por alguien que confunde rojo y verde?

In [ ]:
# La lista, convertida en código: una función que revisa lo que se puede revisar solo.
def revisa_figura(fig, tiene_barras=False):
    """Comprueba automáticamente los puntos de la lista que son verificables.

    No sustituye a la lista —las preguntas 1, 2, 7, 8 y 9 las tiene que contestar una
    persona— pero atrapa los descuidos, que son la mayoría de los fallos.
    """
    avisos = []
    for i, ax in enumerate(fig.axes):
        nombre = f"panel {i}"
        # Un colorbar es un Axes y no tiene por qué llevar etiquetas: se salta.
        if not ax.has_data():
            continue
        # `get_suptitle` existe desde Matplotlib 3.8. El getattr evita depender de
        # `fig._suptitle`, que es interno y puede desaparecer sin avisar.
        suptitulo = getattr(fig, "get_suptitle", lambda: "")()
        if not ax.get_title() and not suptitulo:
            avisos.append(f"{nombre}: sin título ni suptitle")
        if not ax.get_xlabel():
            avisos.append(f"{nombre}: falta la etiqueta del eje X")
        if not ax.get_ylabel():
            avisos.append(f"{nombre}: falta la etiqueta del eje Y")
        for eje, etiqueta in ((ax.get_xlabel(), "X"), (ax.get_ylabel(), "Y")):
            if eje and not any(u in eje for u in
                               ("(", "€", "%", "días", "unidades", "n =")):
                avisos.append(f"{nombre}: la etiqueta del eje {etiqueta} "
                              f"({eje!r}) no lleva unidad")
        if tiene_barras and ax.get_ylim()[0] > 1e-9:
            avisos.append(f"{nombre}: hay barras y el eje Y empieza en "
                          f"{ax.get_ylim()[0]:.2f}, no en cero")
    return avisos


# La probamos con un gráfico descuidado a propósito.
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(["a", "b", "c"], [10, 12, 11])
ax.set_ylim(9, 13)
ax.set_title("ventas")
plt.close(fig)

print("Revisión del gráfico descuidado:")
for aviso in revisa_figura(fig, tiene_barras=True):
    print("  ✗", aviso)

# Y con uno correcto.
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(["a", "b", "c"], [10, 12, 11])
ax.set_ylim(0, 14)
ax.set_title("Las tres tiendas venden lo mismo")
ax.set_xlabel("Tienda")
ax.set_ylabel("Ventas (miles de €)")
plt.close(fig)

avisos = revisa_figura(fig, tiene_barras=True)
print()
print("Revisión del gráfico correcto:")
if avisos:
    for aviso in avisos:
        print("  ✗", aviso)
else:
    print("  ✓ nada que objetar en lo que se puede revisar automáticamente")
print()
print("Lo que esta función NO puede revisar, y es lo importante:")
print("  - si el título es la respuesta a una pregunta o solo una etiqueta,")
print("  - si se está enseñando toda la serie,")
print("  - si la media que se dibuja resume algo de verdad,")
print("  - si hay una variable de confusión detrás de la comparación.")
print()
print("Esas cuatro las contesta una persona, y son las que separan un gráfico")
print("correcto de un gráfico honesto.")

## Ejercicios

### Ejercicio 1. Fabricar el engaño

Con los datos de `ventas`, construye **una figura con seis paneles**: los seis
engaños de las secciones 1 a 7, cada uno sobre datos de TechStore y con el titular
falso que lo acompaña.

Y después, la parte que cuenta: **para cada panel, escribe en una línea qué
afirmación falsa hace y qué decisión de dibujo la produce**.

Este ejercicio es la mitad del criterio: hasta que no se sabe fabricar el engaño no
se sabe detectarlo.

### Ejercicio 2. Detectar el engaño ajeno

La celda siguiente dibuja cuatro gráficos sobre TechStore. **Tres tienen un problema
grave y uno es correcto.**

1. Di cuál es el correcto.
2. Para cada uno de los otros tres, nombra el problema y di qué afirmación falsa
   produce.
3. Redibuja los tres arreglados, con el título convertido en la respuesta a una
   pregunta.

In [ ]:
_muestra = ventas.sample(500, random_state=7)
_por_mes = ventas.groupby("Mes")["Importe"].sum()

fig, ax = plt.subplots(2, 2, figsize=(14, 8))

# (a)
_medias = ventas.groupby("Categoria")["Importe"].mean().sort_values()
ax[0, 0].bar(_medias.index, _medias.to_numpy(), color="#8e44ad")
ax[0, 0].set_ylim(_medias.min() - 5, _medias.max() + 5)
ax[0, 0].set_title("(a) Importe medio por categoría")
ax[0, 0].tick_params(axis="x", rotation=45, labelsize=7)

# (b)
ax[0, 1].plot(_por_mes.index[-5:], _por_mes.to_numpy()[-5:], "o-",
              color="#c0392b", linewidth=3)
ax[0, 1].set_title("(b) Las ventas se desploman")
ax[0, 1].tick_params(axis="x", rotation=45, labelsize=7)

# (c)
_desc = ventas.groupby(pd.cut(ventas["Descuento_%"], [0, 10, 20, 30, 100],
                              include_lowest=True),
                       observed=True)["Cantidad"].mean()
ax[1, 0].scatter(range(len(_desc)), _desc.to_numpy(),
                 s=(_desc.to_numpy() * 60) ** 2, color="#e67e22", alpha=0.7)
ax[1, 0].set_title("(c) Más descuento, más unidades")
ax[1, 0].set_xticks(range(len(_desc)), [str(i) for i in _desc.index], fontsize=7)

# (d)
_reg = ventas.groupby("Region")["Importe"].sum().sort_values(ascending=False)
ax[1, 1].barh(np.arange(len(_reg))[::-1], _reg.to_numpy(), color="#1a5276")
ax[1, 1].set_yticks(np.arange(len(_reg))[::-1], _reg.index, fontsize=8)
ax[1, 1].set_xlim(0, _reg.max() * 1.1)
ax[1, 1].set_xlabel("Importe total (€)")
ax[1, 1].set_ylabel("Región")
ax[1, 1].set_title("Madrid y la Comunitat Valenciana son la mitad del negocio")

fig.tight_layout()
plt.show()

# TODO: Escribe tu análisis y los tres gráficos arreglados

### Ejercicio 3. La media que no resume

Para cada una de las seis categorías de `ventas`:

1. Calcula media, mediana, desviación y los percentiles 5 y 95 del importe.
2. Decide si la media resume bien el reparto, con un criterio **escrito** (por
   ejemplo: la media resume si está a menos de un 10 % de la mediana).
3. Dibuja una figura que enseñe la forma de las seis y en la que se vea de un vistazo
   en cuáles la media engaña.

Y contesta: si tuvieras que dar **una sola cifra** por categoría a la dirección,
¿cuál darías en cada caso, y por qué no la misma en todas?

### Ejercicio 4. Buscar una paradoja de Simpson en TechStore

La sección 6 usa un caso fabricado. Búscalo en los datos reales.

1. Elige una comparación de dos grupos en `ventas`: por ejemplo, el importe medio de
   dos regiones, o de las transacciones con descuento frente a las sin descuento.
2. Desglósala por **categoría**, por **mes** y por **tramo de precio unitario**.
3. Comprueba si el orden se invierte en algún desglose.
4. Si lo encuentras, dibuja los tres paneles de la sección 6: el total, el desglose y
   la causa. Si no lo encuentras, **dilo y demuéstralo**: enseña los desgloses en los
   que el orden se mantiene.

«No lo he encontrado, y aquí está la comprobación» es una respuesta completa. «No lo
he buscado» no.

### Ejercicio 5. El rediseño de los cuatro pasos

Coge **el peor gráfico que hayas hecho** en los cuadernos 01 a 05 —el que menos te
guste— y aplícale los cuatro pasos de la sección 8, dibujando los cuatro:

1. Escribe la pregunta a la que responde, en una frase.
2. Elige la codificación con la tabla de Cleveland y McGill, y justifica la elección.
3. Quita todo lo que no responda a la pregunta, y enumera qué has quitado.
4. Convierte el título en la respuesta, y añade la nota de procedencia.

Después pásale `revisa_figura` y arregla lo que salga.

### Ejercicio 6. Ampliar el revisor

La función `revisa_figura` comprueba cuatro cosas. Añádele tres más:

1. Que ningún panel tenga más de **siete** series o categorías en la leyenda (por
   encima de eso, la leyenda deja de ser legible y hay que repartir en paneles).
2. Que el mapa de color usado no sea `jet` ni `rainbow`. Pista: los objetos de imagen
   de un `Axes` están en `ax.get_images()` y tienen `.get_cmap().name`.
3. Que si hay una barra de color, tenga etiqueta.

Y una cuarta a tu elección, de las que sí se pueden comprobar automáticamente.
Pruébala con dos figuras: una que pase y otra que falle en cada comprobación.

### Ejercicio 7. Una figura, un mensaje

Es el ensayo directo de la parte final de la P4.2.

Con `ventas`, elige **un hallazgo** que te parezca el más importante de TechStore y
produce **una sola figura** que lo sostenga, dirigida a alguien que no programa.

Condiciones:

1. Una figura. Puede tener paneles, pero cuenta **un** mensaje.
2. El título es la afirmación, y la afirmación tiene que ser comprobable con los
   datos. Compruébala en código.
3. La nota de procedencia al pie: de dónde salen los datos, de cuándo son, y qué
   filas se han descartado al limpiar.
4. Pasa la lista de comprobación de las nueve preguntas, y escribe la respuesta a
   cada una.
5. **Un límite declarado**: una cosa que tu figura NO permite concluir y que alguien
   podría creerse al verla.

El punto 5 es el que más pesa en la rúbrica, y es el que casi nadie escribe.

## Lo que hay que llevarse de aquí

1. **Un gráfico es un argumento.** Cada decisión de dibujo es una afirmación sobre
   los datos, y las afirmaciones pueden ser falsas sin que ningún dato lo sea.
2. **La longitud pide el cero; la pendiente, no.** Y en los dos casos hay que poder
   justificar el límite del eje.
3. **Los sectores solo para dos o tres partes de un total con significado.** Si hay
   que escribir el porcentaje, el ángulo no aportaba nada.
4. **El valor va en el área, no en el radio.** En Matplotlib, `s` ya es el área.
5. **Enseñar siempre la serie completa.** Si se destaca un tramo, dentro de la serie
   completa. Es la única defensa contra la ventana escogida, y es de proceso.
6. **Una barra con una media necesita justificación.** Tres repartos que no se
   parecen en nada pueden tener la misma media.
7. **Antes de publicar una comparación, desglósala.** La paradoja de Simpson no la
   detecta ninguna técnica de dibujo: hay que sospecharla.
8. **El acumulado no puede bajar**, así que no sirve para responder «¿vamos bien?».
9. **Tres preguntas, en orden:** a qué responde, con qué codificación, y qué se
   puede quitar. La tercera no se contesta antes que la primera.
10. **El título es la respuesta, no la etiqueta.** Y tiene que ser comprobable.
11. **Un límite declarado vale más que un hallazgo extra.** Es lo que distingue un
    análisis de un folleto.

## Para seguir

- Edward Tufte, *The Visual Display of Quantitative Information* (1983). El libro
  fundacional. De aquí sale la idea de quitar todo lo que no informa.
- William Cleveland y Robert McGill, *Graphical Perception* (1984). El artículo con
  los experimentos de los que sale la tabla de codificaciones de la sección 8.
- Alberto Cairo, *How Charts Lie* (2019). Un libro entero sobre lo que hace este
  cuaderno, con casos de prensa reales.
- [Calling Bullshit](https://www.callingbullshit.org/) — Bergstrom y West, Universidad
  de Washington. El curso está en abierto y la sección de gráficos engañosos es
  directamente material de clase.
- [Data to Viz · el catálogo de errores](https://www.data-to-viz.com/caveats.html) —
  una lista de trampas por tipo de gráfico, con el código para reproducirlas.

---

Con esto se cierra el material de la unidad. Las dos prácticas usan lo de los seis
cuadernos: la **P4.1** mide y decide la herramienta (criterio 1.d) y la **P4.2**
evalúa un modelo mirándolo (criterio 2.e), y las dos se corrigen con esta lista de
comprobación aplicada a cada figura.